# Visualisation 3D des embeddings JEPA (PCA + UMAP)

Projette en **3D** l'embedding **poolé par ECG** (moyenne des tokens — exactement ce que voit la
sonde/tête linéaire) des encodeurs **JEPA gelés** (ViT tiny + CNN). Scènes **interactives** (plotly,
rotatables) avec des **boutons un-contre-tous** : chaque bouton surligne une superclasse (couleur)
sur fond gris. Question : les classes se séparent-elles dans l'espace appris **sans labels** ?

In [ ]:
# === SETUP — à exécuter une fois par session. Suivi en direct dans le terminal. ===
import os, subprocess, shutil, time
from pathlib import Path
import torch

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU  :", gpu or "aucun (CPU)")
print("torch:", torch.__version__, "| cuda dispo :", torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')

# >>> URL de ton repo (repo public, sinon token nécessaire) <<<
REPO_URL    = "https://github.com/JulesV19/cardiac-JEPA"
DRIVE_ROOT  = Path('/content/drive/MyDrive/cardiac-jepa')   # persiste entre sessions
DATA_DRIVE  = DRIVE_ROOT / 'data'                           # les 2 CSV
CACHE_DRIVE = DRIVE_ROOT / 'ptbxl_100hz.npy'                # cache signaux (~1 Go)
CACHE_LOCAL = Path('/content/ptbxl_100hz.npy')              # copie locale = lecture rapide
REPO_DIR    = Path('/content/cardiac-jepa')

# --- code : clone ou pull ---
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q umap-learn                # <- réduction non-linéaire (dépendance en +)
print("code prêt :", REPO_DIR)

# --- données : cache Drive -> disque local (une fois) ---
required = [CACHE_DRIVE, DATA_DRIVE / 'ptbxl_database.csv', DATA_DRIVE / 'scp_statements.csv']
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Manquant sur Drive :\n  " + "\n  ".join(missing))
if not CACHE_LOCAL.exists():
    print("copie du cache (~1 Go, une fois)...", flush=True); t0 = time.time()
    shutil.copy(CACHE_DRIVE, CACHE_LOCAL); print(f"  copié en {time.time()-t0:.0f}s")
os.environ['PTBXL_DATA_DIR'] = str(DATA_DRIVE)
os.environ['PTBXL_CACHE']    = str(CACHE_LOCAL)

from jepa.data import PTBXLDataset
ds = PTBXLDataset('pretrain'); x = ds[0]
print(f"OK — pretrain={len(ds)} ECG | échantillon {tuple(x.shape)} "
      f"(moy={x.mean():.3f} std={x.std():.3f})")

In [ ]:
# === CONFIG + chargement des encodeurs JEPA gelés ===
import numpy as np
import torch
from jepa.jepa import JEPA
from jepa.models import ModelConfig
from jepa.data import SUPERCLASSES
from jepa.probe.features import extract_features

SPLIT   = 'test'                       # fold 10 (jamais vu) — change en 'val'/'pretrain' au besoin
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Runs à visualiser : {nom affiché : dossier du run de pré-entraînement JEPA}
RUNS = {
    'ViT-JEPA':  DRIVE_ROOT / 'runs' / 'tiny_v1',
    'CNN-JEPA':  DRIVE_ROOT / 'runs' / 'cnn_v2',   # <- 'cnn_v1' (1,27M) ou 'cnn_v2' (6,25M)
}

def resolve_ckpt(run_dir: Path) -> Path:
    """best.pt si présent, sinon le ckpt périodique le plus avancé (ckpt_e*.pt / latest.pt)."""
    run_dir = Path(run_dir)
    if (run_dir / 'best.pt').exists():
        return run_dir / 'best.pt'
    periodic = sorted(run_dir.glob('ckpt_e*.pt'),
                      key=lambda p: int(p.stem.split('_e')[1]))
    if periodic:
        return periodic[-1]
    if (run_dir / 'latest.pt').exists():
        return run_dir / 'latest.pt'
    raise FileNotFoundError(f"aucun checkpoint dans {run_dir}")

@torch.no_grad()
def load_target_encoder(ckpt: Path):
    """Reconstruit le JEPA depuis sa config sauvegardée, charge les poids, renvoie le target encoder GELÉ."""
    ck = torch.load(ckpt, map_location='cpu', weights_only=False)
    model = JEPA(ModelConfig(**ck['cfg']['model']))
    model.load_state_dict(ck['model'])
    enc = model.target_encoder.to(device).eval()
    for p in enc.parameters():
        p.requires_grad_(False)
    return enc, ck.get('epoch', '?')

# --- Extraction des features poolées (N, D) + labels multi-hot (N, 5), une fois par run ---
FEATS = {}
for name, run_dir in RUNS.items():
    ckpt = resolve_ckpt(run_dir)
    enc, ep = load_target_encoder(ckpt)
    X, y = extract_features(enc, SPLIT, device, workers=2)
    FEATS[name] = (X, y)
    print(f"{name:9s} <- {ckpt.name} (e{ep}) | features {X.shape}  labels {y.shape}")
print(f"\nsplit={SPLIT}  classes={SUPERCLASSES}")

In [ ]:
# === RÉDUCTION 3D : PCA (linéaire) + UMAP (non-linéaire, précédée d'une PCA-50 pour débruiter) ===
from sklearn.decomposition import PCA
import umap

def embed_3d(X: np.ndarray) -> dict:
    """Renvoie {'PCA': (N,3), 'UMAP': (N,3)}. Features z-scorées par dimension d'abord."""
    Xs = (X - X.mean(0, keepdims=True)) / (X.std(0, keepdims=True) + 1e-6)
    pca3 = PCA(n_components=3, random_state=0).fit_transform(Xs)
    n_pc = min(50, Xs.shape[1])
    pca50 = PCA(n_components=n_pc, random_state=0).fit_transform(Xs)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=3, random_state=0)
    umap3 = reducer.fit_transform(pca50)
    return {'PCA': pca3, 'UMAP': umap3}

EMB = {name: embed_3d(X) for name, (X, y) in FEATS.items()}
print('réductions 3D calculées :', {k: {m: v[m].shape for m in v} for k, v in EMB.items()})

In [ ]:
# === TRACÉ 3D INTERACTIF : une scène rotatable par (modèle × méthode), boutons un-contre-tous ===
import plotly.graph_objects as go

# Palette CVD-safe (Okabe-Ito) : une couleur d'accent par superclasse.
ACCENT = {'NORM': '#009E73', 'MI': '#D55E00', 'STTC': '#0072B2',
          'CD': '#CC79A7', 'HYP': '#E69F00'}

def plot_model_3d(name: str):
    X, y = FEATS[name]
    for m, E in EMB[name].items():
        fig = go.Figure()
        # Fond : tous les ECG en gris (toujours visible).
        fig.add_trace(go.Scatter3d(
            x=E[:, 0], y=E[:, 1], z=E[:, 2], mode='markers', name='tous',
            marker=dict(size=1.8, color='#d9d9d9', opacity=0.35), hoverinfo='skip'))
        # Une trace colorée par classe (positifs). Seule la 1re est visible au départ.
        for j, cls in enumerate(SUPERCLASSES):
            pos = y[:, j] > 0.5
            fig.add_trace(go.Scatter3d(
                x=E[pos, 0], y=E[pos, 1], z=E[pos, 2], mode='markers',
                name=f'{cls} (n={int(pos.sum())})', visible=(j == 0),
                marker=dict(size=3, color=ACCENT[cls], opacity=0.85)))
        # Boutons un-contre-tous : n'affiche qu'une classe colorée à la fois (+ le fond gris).
        buttons = [dict(label=cls, method='update',
                        args=[{'visible': [True] + [k == j for k in range(len(SUPERCLASSES))]}])
                   for j, cls in enumerate(SUPERCLASSES)]
        fig.update_layout(
            title=f'{name} — {m} 3D — embeddings poolés par ECG ({SPLIT}, gelé, un-contre-tous)',
            updatemenus=[dict(type='buttons', direction='right', x=0.5, y=1.06,
                              xanchor='center', showactive=True, buttons=buttons)],
            scene=dict(xaxis_title=f'{m}1', yaxis_title=f'{m}2', zaxis_title=f'{m}3'),
            width=820, height=680, margin=dict(l=0, r=0, t=90, b=0))
        fig.show()

for name in FEATS:
    plot_model_3d(name)